# Hyperion Economy Notebook (Self-Contained)

Dieses Notebook enthält **alles in einer Datei**:
- Datenmodell
- Simulationslogik
- Batch-Simulation über Jahre/Perioden
- Auswertungen mit Pandas + Matplotlib

Kein Import aus `trade_sim.py` nötig.


In [ ]:
import random
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('ggplot')


## 1) Parameter


In [ ]:
YEARS = 40
SEED = 12
EVENT_CHANCE = 0.48
TRADE_INTENSITY = 1.15
FACTION_STRENGTH = 1.1
CORE_FEE = 0.015

TOP_N = 5
SHOW_EVENT_LOG_ROWS = 20


## 2) Modell (vollständig im Notebook)


In [ ]:
GOOD_DATA: Dict[str, Dict[str, float]] = {
    'nahrung': {'base_price': 28, 'volatility': 0.14, 'strategic': 0.70},
    'rohstoffe': {'base_price': 42, 'volatility': 0.20, 'strategic': 0.80},
    'industrieteile': {'base_price': 67, 'volatility': 0.24, 'strategic': 0.85},
    'energie': {'base_price': 56, 'volatility': 0.19, 'strategic': 0.90},
    'luxus': {'base_price': 120, 'volatility': 0.28, 'strategic': 0.50},
    'biotech': {'base_price': 142, 'volatility': 0.30, 'strategic': 0.95},
    'core_daten': {'base_price': 175, 'volatility': 0.26, 'strategic': 0.70},
    'relikte': {'base_price': 260, 'volatility': 0.40, 'strategic': 0.45},
}
GOODS = list(GOOD_DATA.keys())


@dataclass
class World:
    name: str
    category: str
    population_factor: float
    production: Dict[str, float]
    consumption: Dict[str, float]
    stability: float
    prosperity: float
    faction: str
    farcaster: bool
    periphery: bool
    hyperion_special: bool = False
    pilgrim_pull: float = 0.0
    stock: Dict[str, float] = field(default_factory=dict)
    prices: Dict[str, float] = field(default_factory=dict)
    time_debt: float = 0.0
    embargo_ticks: int = 0

    def __post_init__(self):
        for g in GOODS:
            self.stock.setdefault(g, 16.0 + self.population_factor * 10)
            self.prices.setdefault(g, GOOD_DATA[g]['base_price'])


@dataclass
class EventEffect:
    name: str
    ticks_left: int
    modifiers: Dict[str, float]


@dataclass
class Config:
    years: int
    seed: int
    event_chance: float
    trade_intensity: float
    faction_strength: float
    core_fee: float


In [ ]:
class HyperionNotebookSim:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.rng = random.Random(cfg.seed)
        self.tick = 0
        self.worlds = self._build_worlds()
        self.effects: List[EventEffect] = []
        self.events_now: List[str] = []
        self.trade_log: List[str] = []
        self.metrics = {
            'core_signal': 1.0,
            'ouster_threat': 1.0,
            'templar_access': 0.0,
            'hegemony_control': 1.0,
        }

    def _build_worlds(self) -> List[World]:
        return [
            World('Lusus','Kernwelt',1.6,{'luxus':3,'core_daten':2},{'nahrung':4,'energie':2},0.86,0.90,'Hegemonie',True,False),
            World('Tau Ceti','Agrarwelt',1.4,{'nahrung':6,'biotech':1.5},{'energie':2,'industrieteile':2},0.84,0.72,'Hegemonie',True,False),
            World('Maui-Covenant','Templarwelt',1.0,{'biotech':2.2,'nahrung':2},{'energie':1.5,'luxus':1},0.90,0.70,'Templars',True,False),
            World('Marsim','Industriewelt',1.5,{'industrieteile':5,'energie':3},{'rohstoffe':4,'nahrung':2.4},0.77,0.78,'Hegemonie',True,False),
            World('Qom-Riyadh','Datenwelt',1.2,{'core_daten':4.4,'luxus':1.1},{'nahrung':2,'energie':2},0.80,0.83,'TechnoCore',True,False),
            World('Bressia','Grenzwelt',0.9,{'rohstoffe':3.2,'energie':1.4},{'nahrung':2.6,'industrieteile':1.9},0.63,0.50,'Hegemonie',False,True),
            World('Hebron','Pilgerwelt',1.0,{'luxus':1.3},{'nahrung':2.2,'energie':1.8,'biotech':1.6},0.70,0.55,'Hegemonie',False,True,pilgrim_pull=1.2),
            World('Hyperion','Sonderwelt',0.8,{'relikte':0.7,'rohstoffe':1.4},{'nahrung':1.8,'energie':2.1,'luxus':1.4},0.58,0.45,'Neutral',False,True,hyperion_special=True,pilgrim_pull=2.0),
        ]

    def _mods(self):
        m = dict(farcaster_efficiency=1.0, transport_cost_bonus=0.0, risk_bonus=0.0, core_noise=0.08,
                 pilgrim_demand=1.0, production_penalty=0.0, templar_bonus=0.0)
        for e in self.effects:
            for k,v in e.modifiers.items():
                if k in {'farcaster_efficiency','pilgrim_demand'}:
                    m[k] *= v
                else:
                    m[k] += v
        return m

    def _event_roll(self):
        self.events_now = []
        if self.rng.random() > self.cfg.event_chance:
            return
        table = [
            ('farcaster_stoerung',0.16),('ouster_raid',0.16),('pilgerboom',0.14),('sanktionen',0.12),
            ('core_prognosefehler',0.12),('core_optimierung',0.12),('templar_korridor',0.10),('aufstand',0.08)
        ]
        x=self.rng.random(); p=0; sel='core_optimierung'
        for k,w in table:
            p += w
            if x <= p:
                sel = k; break

        if sel=='farcaster_stoerung':
            self.effects.append(EventEffect('Farcaster-Störung',2,{'farcaster_efficiency':0.25,'transport_cost_bonus':3.5}))
            self.events_now.append('Farcaster-Störung trifft Kernwelten')
        elif sel=='ouster_raid':
            t=self.rng.choice([w for w in self.worlds if w.periphery]); t.stability=max(0.2,t.stability-0.08*self.cfg.faction_strength)
            self.effects.append(EventEffect('Ouster-Druck',2,{'risk_bonus':1.8}))
            self.events_now.append(f'Ouster-Raid bei {t.name}')
        elif sel=='pilgerboom':
            self.effects.append(EventEffect('Pilgerboom',3,{'pilgrim_demand':1.45}))
            self.events_now.append('Pilgerströme Richtung Hyperion')
        elif sel=='sanktionen':
            t=self.rng.choice([w for w in self.worlds if w.faction=='Hegemonie']); t.embargo_ticks=2
            self.events_now.append(f'Politische Sanktionen gegen {t.name}')
        elif sel=='core_prognosefehler':
            self.effects.append(EventEffect('Core-Prognosefehler',2,{'core_noise':0.18}))
            self.events_now.append('TechnoCore-Prognosefehler erhöht Volatilität')
        elif sel=='core_optimierung':
            self.effects.append(EventEffect('Core-Optimierung',2,{'core_noise':-0.04,'transport_cost_bonus':-1.2}))
            self.events_now.append('TechnoCore optimiert Handelsnetz')
        elif sel=='templar_korridor':
            self.effects.append(EventEffect('Templar-Korridor',1,{'templar_bonus':0.9}))
            self.events_now.append('Weltenbaum-Sonderroute verfügbar')
        else:
            t=self.rng.choice([w for w in self.worlds if w.periphery]); t.stability=max(0.2,t.stability-0.12)
            self.effects.append(EventEffect('Peripherie-Aufstand',2,{'production_penalty':0.18,'risk_bonus':1.2}))
            self.events_now.append(f'Aufstand auf {t.name}')

    def _produce_consume(self, m):
        for w in self.worlds:
            pen = m['production_penalty'] if w.periphery else m['production_penalty']*0.4
            stability_factor = 0.85 + (w.stability*0.3)
            for g in GOODS:
                w.stock[g] += w.production.get(g,0.0)*stability_factor*(1-pen)
                demand = w.consumption.get(g,0.0)*w.population_factor
                if w.hyperion_special and g in {'luxus','relikte'}:
                    demand *= 1 + (w.pilgrim_pull*0.2)
                if g in {'luxus','relikte'}:
                    demand *= m['pilgrim_demand']
                if w.stock[g] >= demand:
                    w.stock[g] -= demand
                    w.prosperity=min(1.35,w.prosperity+0.002)
                else:
                    sh = demand - w.stock[g]
                    w.stock[g]=0
                    w.stability=max(0.2,w.stability-min(0.015+sh*0.002,0.06))
                    w.prosperity=max(0.2,w.prosperity-min(0.01+sh*0.0015,0.05))

    def _update_prices(self,m):
        for w in self.worlds:
            for g in GOODS:
                base=GOOD_DATA[g]['base_price']; vol=GOOD_DATA[g]['volatility']; strat=GOOD_DATA[g]['strategic']
                demand=w.consumption.get(g,0.0)*max(0.5,w.population_factor)
                available=w.stock[g]+1.0
                imbalance=(demand+1.0)/available
                tension=max(0.8,1.25-w.stability)
                noise=self.rng.uniform(-m['core_noise'],m['core_noise'])*(1.2 if w.periphery else 0.8)
                p=base*(1+vol*(imbalance-1))*(1+strat*(tension-1))*(1+noise)
                w.prices[g]=max(4.0,round(p,2))

    def _route_cost(self,s,t,m):
        if s.farcaster and t.farcaster:
            cost=0.8/max(0.15,m['farcaster_efficiency'])+m['transport_cost_bonus']; td=0.0
        else:
            cost=4.5+m['transport_cost_bonus']+(2.3 if s.periphery or t.periphery else 0.0)+m['risk_bonus']*0.6
            td=0.35+(0.35 if s.periphery or t.periphery else 0.0)
            if m['templar_bonus']>0 and (s.faction=='Templars' or t.faction=='Templars'):
                cost=max(0.7,cost-2.5*m['templar_bonus']); td=max(0.1,td-0.25)
        cost += self.cfg.core_fee*10*self.metrics['core_signal']
        return max(0.2,cost), td

    def _trade(self,m):
        self.trade_log=[]
        cap=6.0*self.cfg.trade_intensity
        for g in GOODS:
            ex=[]; im=[]
            for w in self.worlds:
                reserve=7.5+w.population_factor*2
                surplus=w.stock[g]-reserve; deficit=reserve-w.stock[g]
                if surplus>0.8 and w.embargo_ticks<=0: ex.append((w,surplus))
                if deficit>0.8: im.append((w,deficit))
            ex.sort(key=lambda t:t[0].prices[g]); im.sort(key=lambda t:t[0].prices[g], reverse=True)
            for buyer,deficit in im:
                rem=deficit
                for idx,(seller,sur) in enumerate(ex):
                    if rem<=0.1 or sur<=0.1 or seller is buyer: continue
                    c,td=self._route_cost(seller,buyer,m)
                    margin=buyer.prices[g]-(seller.prices[g]+c)
                    if margin<=0: continue
                    qty=min(sur,rem,cap)
                    seller.stock[g]-=qty; buyer.stock[g]+=qty
                    seller.prosperity=min(1.45,seller.prosperity+0.004*qty)
                    buyer.prosperity=min(1.45,buyer.prosperity+0.003*qty)
                    buyer.time_debt += qty*td
                    ex[idx]=(seller,sur-qty); rem-=qty
                    self.trade_log.append({'good':g,'seller':seller.name,'buyer':buyer.name,'qty':qty,'unit_cost':c,'margin':margin})

    def _post(self):
        for w in self.worlds:
            if w.embargo_ticks>0: w.embargo_ticks-=1
            if w.time_debt>0:
                drag=min(0.03,0.002+w.time_debt*0.0001)
                w.prosperity=max(0.2,w.prosperity-drag)
                w.time_debt*=0.92
            w.prosperity=min(1.5,max(0.2,w.prosperity))

    def _advance_effects(self):
        kept=[]
        for e in self.effects:
            e.ticks_left -= 1
            if e.ticks_left>0: kept.append(e)
        self.effects=kept

    def step(self):
        self.tick += 1
        self._event_roll()
        m=self._mods()
        self.metrics['templar_access']=m['templar_bonus']
        self.metrics['ouster_threat']=1.0+(m['risk_bonus']*0.2)
        self.metrics['core_signal']=max(0.6,1.0+m['core_noise']*0.8)

        self._produce_consume(m)
        self._update_prices(m)
        self._trade(m)
        self._post()
        self._advance_effects()


## 3) Batch-Simulation laufen lassen


In [ ]:
cfg = Config(
    years=YEARS,
    seed=SEED,
    event_chance=EVENT_CHANCE,
    trade_intensity=TRADE_INTENSITY,
    faction_strength=FACTION_STRENGTH,
    core_fee=CORE_FEE,
)
sim = HyperionNotebookSim(cfg)

rows_year=[]
rows_world=[]
rows_price=[]
rows_trade=[]
rows_event=[]

for _ in range(cfg.years):
    sim.step()

    rows_year.append({
        'year': sim.tick,
        'events': '; '.join(sim.events_now) if sim.events_now else 'keine',
        'core_signal': sim.metrics['core_signal'],
        'ouster_threat': sim.metrics['ouster_threat'],
        'templar_access': sim.metrics['templar_access'],
        'trade_count': len(sim.trade_log),
        'trade_volume': sum(t['qty'] for t in sim.trade_log),
        'avg_trade_margin': (sum(t['margin'] for t in sim.trade_log) / len(sim.trade_log)) if sim.trade_log else 0.0,
    })

    for e in sim.events_now:
        rows_event.append({'year': sim.tick, 'event': e})

    for w in sim.worlds:
        gdp_proxy = sum(w.production.get(g,0.0) * w.prices[g] for g in GOODS)
        dependency = sum(max(0.0, w.consumption.get(g,0.0)-w.production.get(g,0.0)) for g in GOODS)
        rows_world.append({
            'year': sim.tick,
            'world': w.name,
            'category': w.category,
            'faction': w.faction,
            'farcaster': w.farcaster,
            'periphery': w.periphery,
            'hyperion_special': w.hyperion_special,
            'stability': w.stability,
            'prosperity': w.prosperity,
            'time_debt': w.time_debt,
            'gdp_proxy': gdp_proxy,
            'dependency_index': dependency,
        })
        for g in GOODS:
            rows_price.append({
                'year': sim.tick,
                'world': w.name,
                'good': g,
                'price': w.prices[g],
                'stock': w.stock[g],
            })

    for t in sim.trade_log:
        rows_trade.append({'year': sim.tick, **t})


df_year = pd.DataFrame(rows_year)
df_world = pd.DataFrame(rows_world)
df_price = pd.DataFrame(rows_price)
df_trade = pd.DataFrame(rows_trade)
df_event = pd.DataFrame(rows_event)

print('Simulation abgeschlossen.')
print('Rows:', len(df_year), len(df_world), len(df_price), len(df_trade), len(df_event))


## 4) Tabellen-Auswertungen


In [ ]:
df_year.head(10)


In [ ]:
# Event-Häufigkeiten
(df_event['event'].value_counts().rename_axis('event').reset_index(name='count') if len(df_event) else pd.DataFrame(columns=['event','count']))


In [ ]:
# Top-Welten im letzten Jahr nach Wohlstand
last_year = df_world['year'].max()
df_top = (df_world[df_world['year']==last_year]
          .sort_values(['prosperity','stability','gdp_proxy'], ascending=False)
          .head(TOP_N))
df_top


In [ ]:
# Preisvolatilität je Gut (Std-Abweichung über alle Welten/Jahre)
price_vol = (df_price.groupby('good', as_index=False)
             .agg(avg_price=('price','mean'), price_std=('price','std'))
             .sort_values('price_std', ascending=False))
price_vol


In [ ]:
# Handelsabhängigkeit Peripherie vs Kern
dep_cmp = (df_world.assign(segment=df_world['periphery'].map({True:'Peripherie', False:'Kern/Farcaster'}))
           .groupby(['year','segment'], as_index=False)
           .agg(avg_dependency=('dependency_index','mean'), avg_time_debt=('time_debt','mean')))
dep_cmp.head(10)


In [ ]:
# Konzentration (HHI) der Wirtschaftsleistung pro Jahr
hhi_rows=[]
for y, grp in df_world.groupby('year'):
    shares = grp['gdp_proxy'] / grp['gdp_proxy'].sum()
    hhi = float((shares**2).sum())
    hhi_rows.append({'year': y, 'hhi_gdp': hhi})
df_hhi = pd.DataFrame(hhi_rows)
df_hhi.head()


## 5) Grafiken


In [ ]:
# 5.1 Wohlstandstrend der Top-Welten
fig, ax = plt.subplots(figsize=(10,5))
for w in df_top['world']:
    g = df_world[df_world['world']==w]
    ax.plot(g['year'], g['prosperity'], label=w)
ax.set_title('Wohlstandsentwicklung Top-Welten')
ax.set_xlabel('Jahr')
ax.set_ylabel('Prosperity')
ax.legend()
plt.show()


In [ ]:
# 5.2 Stabilität Kern vs Peripherie
seg = (df_world.assign(segment=df_world['periphery'].map({True:'Peripherie', False:'Kern/Farcaster'}))
       .groupby(['year','segment'], as_index=False)
       .agg(avg_stability=('stability','mean')))
fig, ax = plt.subplots(figsize=(10,5))
for seg_name, grp in seg.groupby('segment'):
    ax.plot(grp['year'], grp['avg_stability'], label=seg_name)
ax.set_title('Stabilität: Kern vs Peripherie')
ax.set_xlabel('Jahr'); ax.set_ylabel('Ø Stabilität')
ax.legend(); plt.show()


In [ ]:
# 5.3 Fraktionsdruck-Signale
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(df_year['year'], df_year['core_signal'], label='Core-Signal')
ax.plot(df_year['year'], df_year['ouster_threat'], label='Ouster-Druck')
ax.plot(df_year['year'], df_year['templar_access'], label='Templar-Zugang')
ax.set_title('Systemische Fraktionssignale')
ax.set_xlabel('Jahr'); ax.set_ylabel('Index')
ax.legend(); plt.show()


In [ ]:
# 5.4 Durchschnittspreise je Gut
piv = df_price.groupby(['year','good'])['price'].mean().reset_index().pivot(index='year', columns='good', values='price')
fig, ax = plt.subplots(figsize=(12,6))
piv.plot(ax=ax)
ax.set_title('Durchschnittspreise je Gut')
ax.set_xlabel('Jahr'); ax.set_ylabel('Preis')
plt.show()


In [ ]:
# 5.5 Hyperion-Fokus: Reliktpreis + Time Debt
h_relic = df_price[(df_price['world']=='Hyperion') & (df_price['good']=='relikte')]
h_world = df_world[df_world['world']=='Hyperion']
fig, ax1 = plt.subplots(figsize=(10,5))
ax1.plot(h_relic['year'], h_relic['price'], color='purple')
ax1.set_ylabel('Reliktpreis', color='purple')
ax1.set_xlabel('Jahr')
ax2 = ax1.twinx()
ax2.plot(h_world['year'], h_world['time_debt'], color='black', linestyle='--')
ax2.set_ylabel('Time Debt', color='black')
ax1.set_title('Hyperion: Reliktpreis vs Time Debt')
plt.show()


In [ ]:
# 5.6 HHI-Konzentration
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(df_hhi['year'], df_hhi['hhi_gdp'])
ax.set_title('Konzentration der Wirtschaftsleistung (HHI)')
ax.set_xlabel('Jahr'); ax.set_ylabel('HHI')
plt.show()


In [ ]:
# 5.7 Handelsvolumen + Margen
fig, ax1 = plt.subplots(figsize=(10,5))
ax1.plot(df_year['year'], df_year['trade_volume'], label='Handelsvolumen', color='tab:blue')
ax1.set_ylabel('Volumen', color='tab:blue')
ax2 = ax1.twinx()
ax2.plot(df_year['year'], df_year['avg_trade_margin'], label='Ø Marge', color='tab:red')
ax2.set_ylabel('Ø Marge', color='tab:red')
ax1.set_title('Handelsvolumen und Ø Marge')
ax1.set_xlabel('Jahr')
plt.show()


## 6) Risiko- und Sensitivitätssicht


In [ ]:
# Risiko-Ranking: hohe Time Debt + niedrige Stabilität
risk = (df_world[df_world['year']==last_year]
        .assign(risk_score=lambda d: (d['time_debt']*0.6) + ((1-d['stability'])*10)*0.4)
        .sort_values('risk_score', ascending=False)
        [['world','category','stability','time_debt','dependency_index','risk_score']])
risk


In [ ]:
# Event-Timeline (gekürzt)
if len(df_event):
    display(df_event.head(SHOW_EVENT_LOG_ROWS))
else:
    print('Keine Events in dieser Konfiguration ausgelöst.')


## 7) Export (optional)


In [ ]:
# Optional:
# df_year.to_csv('hyperion_year.csv', index=False)
# df_world.to_csv('hyperion_world.csv', index=False)
# df_price.to_csv('hyperion_price.csv', index=False)
# df_trade.to_csv('hyperion_trade.csv', index=False)
# df_event.to_csv('hyperion_event.csv', index=False)
